# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farida596/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pyarrow

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!" if HF_TOKEN else "Token not found!")

Token loaded successfully!


In [3]:
import duckdb

con = duckdb.connect()

con.sql(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [7]:
con.sql("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5;
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

One row represents the daily performance of one content page for one client on one report date.

### Time Window

This notebook uses data from March 2026 (month = 2026-03).

In [8]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Section 2: Data Contract

### Prediction Target

| Item | Description |
|------|-------------|
| **Target** | `refresh_opportunity` (derived label) |
| **Definition** | Indicates whether a page should be refreshed based on its historical performance metrics. |
| **Type** | Binary Classification (Yes / No) |

> **Note:** The dataset does not contain a `refresh_opportunity` column. This label will be created later using business rules based on page performance.

---

### Data Grain

| Item | Description |
|------|-------------|
| **One row represents** | One content page (`content_hash_id`) for one client (`client_hash_id`) on one reporting date (`report_date`). |

---

### Features (Inputs)

| Feature | Purpose |
|---------|---------|
| `gsc_impressions` | Number of times the page appeared in Google Search results. |
| `gsc_clicks` | Number of clicks the page received from Google Search. |
| `gsc_sum_position` | Represents the page's average position in Google Search results. |
| `sessions` | Number of website sessions. |
| `sessions_ai` | Number of website visits originating from AI platforms. |
| `scroll_events` | Measures user engagement with the page. |
| `client_has_gsc` | Indicates whether Google Search Console data is available. |
| `client_has_ga4` | Indicates whether Google Analytics 4 data is available. |
| `gsc_data_available` | Confirms that Google Search Console metrics are available. |
| `ga4_data_available` | Confirms that Google Analytics 4 metrics are available. |

---

### Context Columns

| Column | Purpose |
|--------|---------|
| `report_date` | Indicates when the metrics were collected. |
| `month` | Identifies the reporting month. |
| `client_hash_id` | Identifies the client. |
| `content_hash_id` | Identifies the content page. |

---

### Excluded Columns

| Column | Reason |
|--------|--------|
| `client_hash_id` | Identifier only; not useful for prediction. |
| `content_hash_id` | Identifier only; not useful for prediction. |
| `report_date` | Used for tracking time rather than prediction. |
| `month` | Used for grouping records, not as a predictive feature. |

---

### Assumptions

- Each row represents one content page for one client on one reporting date.
- Google Search Console metrics accurately reflect page search performance.
- Lower search performance may indicate that a page is a good candidate for a content refresh.
- The target label (`refresh_opportunity`) will be generated later using business rules because it is not included in the raw dataset.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# ============================================================
# Section 3: Verify the Data Contract
# ============================================================

DATA_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

# ------------------------------------------------------------
# 1. Total number of rows
# ------------------------------------------------------------
print("========== Total Rows ==========")

display(
    con.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{DATA_PATH}')
    """).df()
)

# ------------------------------------------------------------
# 2. Verify the data grain
# One row = report_date + client_hash_id + content_hash_id
# ------------------------------------------------------------
print("========== Duplicate Grain Check ==========")

display(
    con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_count
    FROM read_parquet('{DATA_PATH}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
    """).df()
)

# ------------------------------------------------------------
# 3. Verify the date range
# ------------------------------------------------------------
print("========== Date Range ==========")

display(
    con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet('{DATA_PATH}')
    """).df()
)

# ------------------------------------------------------------
# 4. Check missing values in important columns
# ------------------------------------------------------------
print("========== Missing Values ==========")

display(
    con.sql(f"""
    SELECT
        SUM(CASE WHEN report_date IS NULL THEN 1 ELSE 0 END) AS missing_report_date,
        SUM(CASE WHEN client_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_client,
        SUM(CASE WHEN content_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_content,
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_clicks
    FROM read_parquet('{DATA_PATH}')
    """).df()
)

print("✅ Data contract verification completed.")

========== Total Rows ==========


,total_rows
0,9841378


========== Duplicate Grain Check ==========


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


...

,start_date,end_date
0,2026-03-01,2026-03-31


========== Missing Values ==========


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_report_date,missing_client,missing_content,missing_impressions,missing_clicks
0,0.0,0.0,0.0,0.0,0.0


✅ Data contract verification completed.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Section 4: Data Limits

Although this dataset is useful for identifying pages that may need a content refresh, it has several limitations:

- The data only covers **March 2026**, so we cannot determine long-term performance trends or seasonality.
- External factors, such as Google algorithm updates, competitor changes, holidays, or marketing campaigns, are not included and may affect page performance.
- Client and content identifiers are anonymized using hash IDs, which protects privacy but prevents direct identification of the pages.
- This dataset can help identify pages that are candidates for refresh, but additional content analysis is required before deciding what changes should be made.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.